# Using genomic-benchmarks-data

This notebook demonstrates dataset discovery, metadata, loading, caching, and quality-control (QC) results with the full package. The full package exposes all published splits, including test data.

Install the package if it is not already available in this environment:

```python
%pip install genomic-benchmarks-data
```

In [ ]:
from html import escape
from IPython.display import HTML, display
import genomic_benchmarks_data as gbd


def display_qc_report(path, height=900):
    """Display a standalone QC report without leaking its styles into Jupyter."""
    document = escape(path.read_text(encoding="utf-8"), quote=True)
    return HTML(
        f"<iframe srcdoc=\"{document}\" "
        f"style=\"width: 100%; height: {height}px; border: 1px solid #ddd;\""
        "></iframe>"
    )

## Discover datasets

In [ ]:
collections = gbd.list_collections()
binary_datasets = gbd.list_datasets(task_type="Binary classification")

collections, binary_datasets

### Select the core set

The core set is a selected subset of benchmarks. Pass `core_set=True` to select it, or `core_set=False` to select its complement. This can be combined with the other dataset filters.

In [ ]:
core_datasets = gbd.list_datasets(core_set=True)
core_binary_datasets = gbd.list_datasets(
    core_set=True,
    task_type="Binary classification",
)

{"all_core": core_datasets, "binary_core": core_binary_datasets}

The examples below use a small dataset that provides train, validation (`val`), and test splits.

In [ ]:
COLLECTION = "BEACON"
DATASET = "CRISPROnTarget"

## Inspect metadata

In [ ]:
dataset_info = gbd.get_dataset_info(COLLECTION, DATASET)
train_info = gbd.get_split_info(COLLECTION, DATASET, "train")

{
    "task_type": dataset_info["task_type"],
    "core_set": dataset_info["core_set"],
    "sequence_columns": dataset_info["seq_col"],
    "label_column": dataset_info["label_col"],
    "available_splits": dataset_info["splits"],
    "train_size": train_info["dataset_size"],
}

## Load data

By default, `train` and `val` contain the sequence and label columns needed for the task.

In [ ]:
train = gbd.load_dataset(COLLECTION, DATASET, "train")
validation = gbd.load_dataset(COLLECTION, DATASET, "val")

{"train_columns": train.columns.tolist(), "val_columns": validation.columns.tolist()}

In [ ]:
train.head()

Use `columns="all"` when auxiliary columns are needed.

In [ ]:
train_all = gbd.load_dataset(COLLECTION, DATASET, "train", columns="all")
train_all.columns.tolist()

The default `test` view starts with `id` and `row_index`, followed by the sequence and label columns. These identifiers make predictions traceable to examples.

In [ ]:
test = gbd.load_dataset(COLLECTION, DATASET, "test")
test.head()

## Locate a downloaded file

`download_dataset()` returns the complete cached file, regardless of the column view selected by `load_dataset()`.

In [ ]:
cached_test_path = gbd.download_dataset(COLLECTION, DATASET, "test")
cached_test_path

## Explore QC results

Class QC examines individual splits. Split QC compares two splits for leakage.

In [ ]:
class_qc = gbd.load_qc_results("class")
split_qc = gbd.load_qc_results("split")

{
    "class_checks": gbd.list_qc_metrics("class"),
    "split_checks": gbd.list_qc_metrics("split"),
}

In [ ]:
passing_class_qc = gbd.filter_datasets_by_qc("class", flag="Pass")
passing_split_qc = gbd.filter_datasets_by_qc("split", flag="Pass")

{"class": passing_class_qc, "split": passing_split_qc}

In [ ]:
gbd.plot_qc_summary("class")

## Open self-contained QC reports

In [ ]:
class_report = gbd.get_class_qc_report(
    COLLECTION, DATASET, split="train", seq_col="seq"
)
display(display_qc_report(class_report))

In [ ]:
split_report = gbd.get_split_qc_report(
    COLLECTION, DATASET, split_a="train", split_b="test"
)
display(display_qc_report(split_report))